# Clean and format data for general use: HK1-R12Ter
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from itertools import product

import pandas as pd

from hk1_r12ter_analysis.config import (
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    logger,
)
from hk1_r12ter_analysis.data.cleaning import (
    remove_constant_features,
    replace_zero_values_with_nan,
)
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Load data and filter for groups of interest

In [ ]:
sample_key = "Sample"
group_key = "Group"
groups_of_interest = ["Control", "HK1"]

constant_feature_tolerance = 1e-12
axis = 0

### Load sample metadata

In [ ]:
filename = make_filename("Sample", "Metadata")
input_dirpath = RAW_DATA_DIR / "HK1-R12Ter"

df_sample_meta = load_dataframe_from_file(input_dirpath / filename, index_col=sample_key)
df_sample_meta

### Filter data for groups of interest

In [ ]:
source_types = ["RBC", "Plasma"]
data_types = ["Proteins", "Metabolites", "Lipids", "Oxylipins"]
value_type = "Intensities"

# Directory paths
input_dirpath = RAW_DATA_DIR / "HK1-R12Ter"
output_dirpath = INTERIM_DATA_DIR / "HK1-R12Ter" / "Formatted"
output_dirpath.mkdir(parents=True, exist_ok=True)


for source_type, data_type in product(source_types, data_types):
    # Load data
    logger.debug(f"Processing data for '{source_type}-{data_type}'...")
    filename_args = (source_type, data_type, value_type)
    filename = make_filename(*filename_args)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=sample_key)

    # Add group data
    logger.debug(
        f"Adding groups metadata to '{source_type}-{data_type}' data and filtering for groups of interest: {groups_of_interest}..."
    )
    df[group_key] = df_sample_meta.loc[list(df.index), group_key]
    df = df.set_index([group_key], append=True)
    # Filter for groups of interest
    df = df.loc[df.index.get_level_values(group_key).isin(groups_of_interest)]
    logger.info(
        f"Added groups metadata and filtered '{source_type}-{data_type}' data for groups of interest: {groups_of_interest}."
    )

    # Zero values are considered NA as they are not present/below detection limit in the data.
    df = replace_zero_values_with_nan(df)
    # NA or constant values across features create numerical issues
    df = remove_constant_features(df, tolerance=constant_feature_tolerance, axis=axis)
    # Save data
    save_dataframe_to_file(df, output_dirpath / filename, index=True)
    logger.info(f"Processed data for '{source_type}-{data_type}'.")

### Metadata for annotation

In [ ]:
value_type = "Metadata"
# Directory paths
input_dirpath = RAW_DATA_DIR / "HK1-R12Ter"
output_dirpath = INTERIM_DATA_DIR / "HK1-R12Ter"
output_dirpath.mkdir(parents=True, exist_ok=True)


data_types = [
    "Metabolites",
    "Lipids",
    "Oxylipins",
    "Proteins",
]
for data_type in data_types:
    logger.debug(f"Processing data for '{data_type}-{value_type}'...")
    filename_args = (data_type, value_type)
    filename = make_filename(*filename_args)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=0)
    if data_type != "Proteins":
        df = df[["Omics Lipid Class" if data_type == "Lipids" else "Omics Pathway"]]
    save_dataframe_to_file(df, output_dirpath / filename, index=True)
    logger.info(f"Processed data for '{data_type}-{value_type}'.")

### Physiological and Redox Data

In [ ]:
data_type = "Physiological-Redox"
filename = make_filename(data_type, "data")
# Directory paths
input_dirpath = RAW_DATA_DIR / "HK1-R12Ter"
output_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / data_type
output_dirpath.mkdir(parents=True, exist_ok=True)

# Import data
logger.debug(f"Processing '{data_type}' data...")
df = load_dataframe_from_file(input_dirpath / filename, index_col=0)
df = df.convert_dtypes()

# Format data for groups of interest
logger.debug(
    f"Formatting '{data_type}' and filtering for groups of interest: {groups_of_interest}..."
)
df.index = pd.MultiIndex.from_tuples(list(df.index.str.split("_")), names=["Group", "#"])
df.columns = pd.MultiIndex.from_tuples(
    [x[0] for x in df.columns.str.findall(r"(.*)\W\((.*)\)")], names=["Variable", "Unit"]
)
df = df.loc[groups_of_interest]
logger.info(
    f"Formatted '{data_type}' data and filtering for groups of interest: {groups_of_interest}."
)

# Export data
save_dataframe_to_file(df, output_dirpath / filename, index=True)
logger.info(f"Processed '{data_type}' data.")